# Filter Runs 

In [19]:
import os

# Print current working directory
print(os.getcwd())

# Change to your desired directory
os.chdir('/Users/svasquez/driblab/dl-vision')
print(os.getcwd())

/Users/svasquez/driblab/dl-vision
/Users/svasquez/driblab/dl-vision


In [20]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [21]:
runs_path = f"stats/obso/datasets/training_runs.jsonl"
runs_path = f"stats/obso/datasets/testing_runs.jsonl"


# Load the JSONL file
runs = []
with open(runs_path, 'r') as f:
    for line in f:
        runs.append(json.loads(line))


print(runs[0])
print(f"Run Length: {len(runs)}")

{'match_id': 622090, 'run_id': '622090_0', 'team_id': 4275, 'player_id': 1248950, 'ball_carrier_id_start': 1248923, 'ball_carrier_id_end': 81306, 'start_frame': 70, 'end_frame': 104, 'start_timestamp': 7.22, 'end_timestamp': 10.62, 'period': 1, 'runner_info': {'start_position': {'x': 66.7, 'y': 40.03}, 'positions': [{'x': 66.7, 'y': 40.03, 'frame': 70, 'period': 1}], 'metrics': {'xR_start': 0.4560000002384186, 'xR_end': 0.3799999952316284, 'PC_start': 0.828, 'PC_end': 0.57, 'xG_start': 0.125, 'xG_end': 0.107, 'OBSO_start': None, 'OBSO_end': None}, 'end_position': {'x': 87.16, 'y': 35.04}}, 'possession': {'start_team': '4275', 'end_team': '4275'}, 'off_ball_teammates': [{'player_id': 1423200, 'start_position': {'x': 97.41, 'y': 64.27}, 'end_position': {'x': 98.12, 'y': 64.29}, 'metrics': {'xR_start': 0.4580000042915344, 'xR_end': 0.4300000071525574, 'PC_start': 0.5, 'PC_end': 0.056, 'xG_start': 0.153, 'xG_end': 0.121, 'OBSO_start': None, 'OBSO_end': None}}, {'player_id': 189024, 'start_

## Function to check if the runs has all the metrics calculated

In [22]:
print(runs[0])

def check_run_metrics(run):

    # Check runner metrics
    xR_runner_start = run['runner_info']['metrics']['xR_start']
    xR_runner_end = run['runner_info']['metrics']['xR_end']

    xG_runner_start = run['runner_info']['metrics']['xG_start']
    xG_runner_end = run['runner_info']['metrics']['xG_end']

    PC_runner_start = run['runner_info']['metrics']['PC_start']
    PC_runner_end = run['runner_info']['metrics']['PC_end']


    # Check if runner metrics are None
    if xR_runner_start is None or xR_runner_end is None or xG_runner_start is None or xG_runner_end is None or PC_runner_start is None or PC_runner_end is None:
        return False



    for plyr in run['off_ball_teammates']:
        

        plyr_metrics = plyr['metrics']

        xR_start = plyr_metrics['xR_start']
        xR_end = plyr_metrics['xR_end']

        xG_start = plyr_metrics['xG_start']
        xG_end = plyr_metrics['xG_end']

        PC_start = plyr_metrics['PC_start']
        PC_end = plyr_metrics['PC_end']

        if xR_start is None or xR_end is None or xG_start is None or xG_end is None or PC_start is None or PC_end is None:

            return False

    return True
    
invalid_runs = []

for run in runs:
    if not check_run_metrics(run):
        invalid_runs.append(run)

print(f"Invalid Runs: {len(invalid_runs)}")

{'match_id': 622090, 'run_id': '622090_0', 'team_id': 4275, 'player_id': 1248950, 'ball_carrier_id_start': 1248923, 'ball_carrier_id_end': 81306, 'start_frame': 70, 'end_frame': 104, 'start_timestamp': 7.22, 'end_timestamp': 10.62, 'period': 1, 'runner_info': {'start_position': {'x': 66.7, 'y': 40.03}, 'positions': [{'x': 66.7, 'y': 40.03, 'frame': 70, 'period': 1}], 'metrics': {'xR_start': 0.4560000002384186, 'xR_end': 0.3799999952316284, 'PC_start': 0.828, 'PC_end': 0.57, 'xG_start': 0.125, 'xG_end': 0.107, 'OBSO_start': None, 'OBSO_end': None}, 'end_position': {'x': 87.16, 'y': 35.04}}, 'possession': {'start_team': '4275', 'end_team': '4275'}, 'off_ball_teammates': [{'player_id': 1423200, 'start_position': {'x': 97.41, 'y': 64.27}, 'end_position': {'x': 98.12, 'y': 64.29}, 'metrics': {'xR_start': 0.4580000042915344, 'xR_end': 0.4300000071525574, 'PC_start': 0.5, 'PC_end': 0.056, 'xG_start': 0.153, 'xG_end': 0.121, 'OBSO_start': None, 'OBSO_end': None}}, {'player_id': 189024, 'start_

## Load necesary match data 

In [23]:
matches_df = pd.read_csv("/Users/svasquez/AWS_PC/loaded_matches_v2.csv")

# Make dict of match_id to side_attack
match_id_to_side_attack = {}

for index, row in matches_df.iterrows():
    match_id_to_side_attack[row['match_id']] = {
        'side_attack': row['side_attack'],
        'home_team_id': row['home_team_id'],
        'away_team_id': row['away_team_id'],
    }

match_id_to_side_attack


{622090: {'side_attack': True, 'home_team_id': 4273, 'away_team_id': 4275},
 622092: {'side_attack': True, 'home_team_id': 4272, 'away_team_id': 1},
 622093: {'side_attack': True, 'home_team_id': 14350, 'away_team_id': 4277},
 622094: {'side_attack': False, 'home_team_id': 20079, 'away_team_id': 4265},
 622096: {'side_attack': True, 'home_team_id': 4270, 'away_team_id': 4274},
 622098: {'side_attack': False, 'home_team_id': 4279, 'away_team_id': 4270},
 622100: {'side_attack': True, 'home_team_id': 61410, 'away_team_id': 14350},
 622101: {'side_attack': True, 'home_team_id': 61409, 'away_team_id': 4266},
 622102: {'side_attack': True, 'home_team_id': 4277, 'away_team_id': 4272},
 622104: {'side_attack': True, 'home_team_id': 4274, 'away_team_id': 4275},
 622106: {'side_attack': True, 'home_team_id': 1, 'away_team_id': 20076},
 622107: {'side_attack': False, 'home_team_id': 1, 'away_team_id': 4264},
 622109: {'side_attack': True, 'home_team_id': 4273, 'away_team_id': 61409},
 622112: {'

## Define Function to extract attack goal coordinates

In [24]:
def get_attack_goal(match_id, team_id, side_attack, period:int, match_id_to_side_attack):

    # Determine if the team id is home or away
    is_home = team_id == match_id_to_side_attack[match_id]['home_team_id']

    attacking_side = None
    
    if side_attack == True:
        if period == 1:
            attacking_side =(0, 34) 
        else:
            attacking_side = (105,34)
    else:
        if period == 1:
            attacking_side = (105,34)
        else:
            attacking_side = (0, 34) 

    # Flip if the match_side is away
    if is_home == False:
        if attacking_side[0] == 0:
            attacking_side = (105,34)
        else:
            attacking_side = (0, 34) 

    return attacking_side


## Augment Runs by calculating additional features

In [15]:
import math

aug_runs = []

for run in runs:
    # Get side_attack from match_id_to_side_attack
    side_attack = match_id_to_side_attack[run['match_id']]

    att_goal = get_attack_goal(run['match_id'], run['team_id'], side_attack, run['period'], match_id_to_side_attack)

    run['att_goal'] = att_goal

    # Add distance to attack goal at the start of the run

    runner_coord = run['runner_info']['start_position']

    dist_to_att_goal_start = math.sqrt((runner_coord['x'] - att_goal[0])**2 + (runner_coord['y'] - att_goal[1])**2)

    run['dist_to_att_goal_start'] = dist_to_att_goal_start

    # Add distance to attack goal at the end of the run

    runner_coord = run['runner_info']['end_position']

    dist_to_att_goal_end = math.sqrt((runner_coord['x'] - att_goal[0])**2 + (runner_coord['y'] - att_goal[1])**2)

    run['dist_to_att_goal_end'] = dist_to_att_goal_end

    # Add diff in distance to attack goal

    run['dist_to_att_goal_diff'] = dist_to_att_goal_end - dist_to_att_goal_start

    # Also add the x-axis distance to the attack goal

    run['x_axis_dist_to_att_goal_start'] = abs(att_goal[0] - run['runner_info']['start_position']['x'])

    run['x_axis_dist_to_att_goal_end'] = abs(att_goal[0] - run['runner_info']['end_position']['x'])

    run['x_axis_dist_to_att_goal_diff'] = run['x_axis_dist_to_att_goal_end'] - run['x_axis_dist_to_att_goal_start']
    
    

    aug_runs.append(run)




## Filter runs in the attacking half and that at least 10% towards the attack goal

In [26]:

filtered_runs = []

for run in aug_runs:

    # get start distance to attack goal
    start_dist_to_att_goal = run['dist_to_att_goal_start']

    # get end distance to attack goal
    end_dist_to_att_goal = run['dist_to_att_goal_end']


    

    # Determine if the run is 10% towards the attack goal
    ten_percent_closer = start_dist_to_att_goal * 0.10

    required_distance = start_dist_to_att_goal - ten_percent_closer

    if end_dist_to_att_goal <= required_distance and run['x_axis_dist_to_att_goal_end'] < 35 and  run['x_axis_dist_to_att_goal_start'] < 52.5 :
        filtered_runs.append(run)


print(f"ORIGINAL LENGTH: {len(runs)}")
print(f"Filtered LENGTH: {len(filtered_runs)}")
print(f"% of runs filtered: {(len(filtered_runs) / len(runs)) * 100}%")







ORIGINAL LENGTH: 82199
Filtered LENGTH: 13169
% of runs filtered: 16.02087616637672%


# Save Filtered Runs

In [27]:

with open('stats/obso/datasets/test_filtered_runs.jsonl', 'w') as f:
    for run in filtered_runs:
        f.write(json.dumps(run) + '\n')



